# Searching 20-newsgroups 

In [1]:
import thematic_search as ts
import numpy as np
import pandas as pd

newsgroups_df = pd.read_parquet("hf://datasets/lmcinnes/20newsgroups_embedded/data/train-00000-of-00001.parquet")
newsgroups_df.head(1)

,post,newsgroup,embedding,map
0,\n\nI am sure some bashers of Pens fans are pr...,rec.sport.hockey,"[-0.04380008950829506, 0.08495834469795227, -0...","[-0.13199903070926666, 10.1972017288208]"


In [2]:
tags = np.unique(newsgroups_df['newsgroup'].to_numpy())

from collections import defaultdict
def build_tree(paths):
    tree = defaultdict(set)
    tree["root"]
    for p in paths:
        parts = p.split(".")
        for i in range(len(parts)):
            node = ".".join(parts[:i+1])
            parent = "root" if i == 0 else ".".join(parts[:i])
            tree[parent].add(node)
            tree[node]
    return {k: sorted(v) for k, v in tree.items()}

tree = build_tree(tags)
cluster_tree, cluster_labels = ts.utils.convert_string_tree(tree) 
ts.utils.print_tree(cluster_tree, cluster_labels=cluster_labels)

root
--alt
----alt.atheism
--comp
----comp.graphics
----comp.os
------comp.os.ms-windows
--------comp.os.ms-windows.misc
----comp.sys
------comp.sys.ibm
--------comp.sys.ibm.pc
----------comp.sys.ibm.pc.hardware
------comp.sys.mac
--------comp.sys.mac.hardware
----comp.windows
------comp.windows.x
--misc
----misc.forsale
--rec
----rec.autos
----rec.motorcycles
----rec.sport
------rec.sport.baseball
------rec.sport.hockey
--sci
----sci.crypt
----sci.electronics
----sci.med
----sci.space
--soc
----soc.religion
------soc.religion.christian
--talk
----talk.politics
------talk.politics.guns
------talk.politics.mideast
------talk.politics.misc
----talk.religion
------talk.religion.misc


In [3]:
tag_to_tuple = {v:k for k,v in cluster_labels.items()}

data = []
for tag in tree.keys():
    layer, cluster = tag_to_tuple[tag]
    data.append({
        'uid':ts.utils.topic_uid((layer,cluster)),
        'name':tag,
        'layer':layer,
        'cluster':cluster,
    })

topic_df = pd.DataFrame(data)
topic_df.head(2)

,uid,name,layer,cluster
0,ABQB,root,5,0
1,AAQB,alt,1,0


In [4]:

cluster_matrices = []
n_layers = max(topic_df['layer'].values) # Don't count the root
n_samples = len(newsgroups_df)
for l in range(n_layers):
    tags_in_layer = topic_df[topic_df['layer']==l]['cluster'].to_list()
    n_tags = len(tags_in_layer)
    matrix = np.zeros((n_samples, n_tags))
    for tag in tags_in_layer:
        tag_name = cluster_labels[(l, tag)]
        matrix[:, tag] =  newsgroups_df['newsgroup'].apply(
            lambda x: x == tag_name
        )
    cluster_matrices.append(matrix)

for matrix in cluster_matrices:
    print(matrix.shape)


(18170, 20)
(18170, 11)
(18170, 5)
(18170, 1)
(18170, 1)


In [5]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("sentence-transformers/all-mpnet-base-v2")

soft_cluster_tree = ts.SoftClusterTree(
    cluster_matrices,
    cluster_tree,
)
topic_db = ts.TopicDatabase(
    soft_cluster_tree = soft_cluster_tree,
    embedding_vectors = np.stack(newsgroups_df['embedding'].values),
    reduced_vectors = np.stack(newsgroups_df['map'].values),
    document_df = newsgroups_df[['post', 'newsgroup']],
    topic_df = topic_df,
    embedding_model = model,
)
topic_db

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

MPNetModel LOAD REPORT from: sentence-transformers/all-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [8]:
topic_db.q.topic_name('alt.atheism').inside().documents()

,post,newsgroup
14,"\n\n\tThere is no notion of heliocentric, or e...",alt.atheism
29,[ . . .]\n\nI am a relativist who would like t...,alt.atheism
56,"\n\nOh, Bobby. You're priceless. Did I ever te...",alt.atheism
136,\n\nOr he was just convinced by religious fant...,alt.atheism
168,"\nActually, I've got an entire list of books w...",alt.atheism
...,...,...
18088,Here's a suggestion for the logical argument F...,alt.atheism
18102,\n\nThat's right. Humans have gone somewhat b...,alt.atheism
18136,Archive-name: atheism/resources\nAlt-atheism-a...,alt.atheism
18142,\n\n\nI'm sure there are many people who work ...,alt.atheism
